In [1]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import utility functions
from utils import (
    analyze_dataframe, 
    plot_dataframe, 
    plot_numerical_target,
    calculate_relationship
)

# Import statistical test functions
from stat_utils import statistical_tests_step1, statistical_tests_step2

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
df = pd.read_csv("artifacts/train_data.csv")

target_column = "잔액_리볼빙일시불이월_target"

In [3]:
# Get all columns except the target column for analysis
analysis_columns = [col for col in df.columns if col != target_column]

print(f"Total columns to analyze: {len(analysis_columns)}")
print(f"Target column: {target_column}")
print(f"Target unique values: {df[target_column].nunique()}")

Total columns to analyze: 226
Target column: 잔액_리볼빙일시불이월_target
Target unique values: 79209


In [4]:
# Perform statistical tests to identify statistically significant columns
# Since target has 79,209 unique values (continuous), use statistical_tests_step2
print("=" * 80)
print("Performing statistical tests for CONTINUOUS target...")
print("=" * 80)

test_results, significant_columns = statistical_tests_step2(df, target_column, analysis_columns)

print("\n" + "=" * 80)
print(f"Total significant columns found: {len(significant_columns)}")
print("=" * 80)

Performing statistical tests for CONTINUOUS target...
Relevant : 연령 (H=797.7375, p=3.5706e-170)
Relevant : VIP등급코드 (H=1186.8644, p=1.1214e-255)
Relevant : 최상위카드등급코드 (H=1688.3832, p=0.0000e+00)
Relevant : 거주시도명 (H=385.8542, p=3.3428e-72)
Relevant : 직장시도명 (H=541.4664, p=5.7380e-105)
Relevant : 가입통신회사코드 (H=194.3100, p=6.3991e-43)
Relevant : Life_Stage (H=1049.5421, p=1.7194e-223)
Relevant : RV약정청구율 (Pearson r=-0.5800, Spearman ρ=-0.6407, abs_corr_threshold=0.5)
Relevant : RV전환가능여부 (H=524.3995, p=4.6706e-116)
Relevant : 잔액_B0M (Pearson r=0.4504, Spearman ρ=0.5164, abs_corr_threshold=0.5)
Relevant : 잔액_일시불_B0M (Pearson r=0.5647, Spearman ρ=0.6761, abs_corr_threshold=0.5)
Relevant : 잔액_리볼빙일시불이월_B0M (Pearson r=0.8188, Spearman ρ=0.8547, abs_corr_threshold=0.5)
Relevant : 월중평잔_일시불_B0M (Pearson r=0.5503, Spearman ρ=0.6501, abs_corr_threshold=0.5)
Relevant : 잔액_일시불_B1M (Pearson r=0.5639, Spearman ρ=0.6614, abs_corr_threshold=0.5)
Relevant : 잔액_일시불_B2M (Pearson r=0.5645, Spearman ρ=0.6605, abs_co

In [7]:
from utils import plot_dataframe

In [9]:
plot_dataframe(df[significant_columns+[target_column]], color_column='잔액_리볼빙일시불이월_target', save_dir='viz/stat_significants')

Dashboard created: viz/stat_significants/dashboard.html
Open this file in your browser to view all 25 plots


In [10]:
significant_columns

['연령',
 'VIP등급코드',
 '최상위카드등급코드',
 '거주시도명',
 '직장시도명',
 '가입통신회사코드',
 'Life_Stage',
 'RV약정청구율',
 'RV전환가능여부',
 '잔액_B0M',
 '잔액_일시불_B0M',
 '잔액_리볼빙일시불이월_B0M',
 '월중평잔_일시불_B0M',
 '잔액_일시불_B1M',
 '잔액_일시불_B2M',
 'RV_평균잔액_R12M',
 'RV_최대잔액_R12M',
 'RV_평균잔액_R6M',
 'RV_최대잔액_R6M',
 'RV_평균잔액_R3M',
 'RV_최대잔액_R3M',
 '월중평잔_일시불',
 '월중평잔_RV일시불',
 '평잔_일시불_3M',
 '평잔_RV일시불_3M']

---

계산 후 남은 x와 통계적으로 유의미한 관계를 지니는 변수 탐색

In [7]:
df['연령'].unique()

array(['40대', '30대', '60대', '50대', '20대', '70대이상'], dtype=object)

In [12]:
b0m = df['잔액_리볼빙일시불이월_B0M'].clip(lower=0)
avg_rv = df['월중평잔_RV일시불'].clip(lower=0)

# 월이율
apr = df['RV일시불이자율_할인전'] * 0.01
monthly_rate = (apr / 12).clip(lower=0)

interest_amount = avg_rv * monthly_rate

# 결제율(%) -> 소수, 범위 클립
pay_rate = (df[['RV약정청구율', 'RV최소결제비율']].max(axis=1) * 0.01).clip(0, 1)

# 이번달 추가 RV 유입
# n/a

# 최소 결제 금액 : (b0m + interest)만 사용 (유입 n/a)
min_pay_amount = pay_rate * (b0m + interest_amount)

# 청구기준 근사치: 최소 결제 금액만 
a1m_hat = (b0m + interest_amount - min_pay_amount).clip(lower=0)

df['a1m_hat'] = a1m_hat

In [13]:
analysis_columns = [i for i in df.columns if i not in ['잔액_리볼빙일시불이월_target']]
target_column = 'a1m_hat'

print("=" * 80)
print("Performing statistical tests for CONTINUOUS target...")
print("=" * 80)

test_results_res, significant_columns_res = statistical_tests_step2(df, target_column, analysis_columns)

print("\n" + "=" * 80)
print(f"Total significant columns found: {len(significant_columns_res)}")
print("=" * 80)

Performing statistical tests for CONTINUOUS target...
Relevant : 연령 (H=745.8540, p=5.9622e-159)
Relevant : VIP등급코드 (H=577.0911, p=1.4060e-123)
Relevant : 최상위카드등급코드 (H=1791.5397, p=0.0000e+00)
Relevant : 거주시도명 (H=341.3623, p=6.5323e-63)
Relevant : 직장시도명 (H=476.8940, p=2.4887e-91)
Relevant : 가입통신회사코드 (H=177.2915, p=3.1743e-39)
Relevant : Life_Stage (H=1021.9013, p=1.6383e-217)
Relevant : RV약정청구율 (Pearson r=-0.8699, Spearman ρ=-0.8109, abs_corr_threshold=0.5)
Relevant : RV전환가능여부 (H=310.2045, p=1.9710e-69)
Relevant : 잔액_일시불_B0M (Pearson r=0.4749, Spearman ρ=0.6064, abs_corr_threshold=0.5)
Relevant : 잔액_리볼빙일시불이월_B0M (Pearson r=0.8894, Spearman ρ=0.9568, abs_corr_threshold=0.5)
Relevant : 월중평잔_일시불_B0M (Pearson r=0.4730, Spearman ρ=0.5891, abs_corr_threshold=0.5)
Relevant : 잔액_일시불_B1M (Pearson r=0.4835, Spearman ρ=0.6020, abs_corr_threshold=0.5)
Relevant : 잔액_일시불_B2M (Pearson r=0.4891, Spearman ρ=0.6067, abs_corr_threshold=0.5)
Relevant : RV_평균잔액_R12M (Pearson r=0.7883, Spearman ρ=0.8858, abs

In [14]:
plot_dataframe(df[significant_columns_res+[target_column]], color_column='a1m_hat', save_dir='viz/stat_significants_res')

Dashboard created: viz/stat_significants_res/dashboard.html
Open this file in your browser to view all 24 plots


In [15]:
significant_columns_res

['연령',
 'VIP등급코드',
 '최상위카드등급코드',
 '거주시도명',
 '직장시도명',
 '가입통신회사코드',
 'Life_Stage',
 'RV약정청구율',
 'RV전환가능여부',
 '잔액_일시불_B0M',
 '잔액_리볼빙일시불이월_B0M',
 '월중평잔_일시불_B0M',
 '잔액_일시불_B1M',
 '잔액_일시불_B2M',
 'RV_평균잔액_R12M',
 'RV_최대잔액_R12M',
 'RV_평균잔액_R6M',
 'RV_최대잔액_R6M',
 'RV_평균잔액_R3M',
 'RV_최대잔액_R3M',
 '월중평잔_일시불',
 '월중평잔_RV일시불',
 '평잔_일시불_3M',
 '평잔_RV일시불_3M']

In [16]:
[i for i in significant_columns if i not in significant_columns_res]

['잔액_B0M']

In [17]:
[i for i in significant_columns_res if i not in significant_columns]

[]

In [ ]:
# Display the list of statistically significant columns
print("Statistically Significant Columns:")
print("-" * 80)
for i, col in enumerate(significant_columns, 1):
    print(f"{i}. {col}")
    
# Save to a list for further analysis
print(f"\n\nVariable name: significant_columns")
print(f"Length: {len(significant_columns)}")

In [ ]:
# Create a summary DataFrame of test results for better visualization
summary_data = []

for var, results in test_results.items():
    is_significant = var in significant_columns
    
    if 'test_type' in results:
        if results['test_type'] == 'correlation':  # Numerical feature (correlation)
            summary_data.append({
                'Column': var,
                'Test Type': 'Correlation (Pearson & Spearman)',
                'Pearson_r': results['pearson_corr'],
                'Pearson_p': results['pearson_p'],
                'Spearman_ρ': results['spearman_corr'],
                'Spearman_p': results['spearman_p'],
                'Significant': is_significant
            })
        elif results['test_type'] == 'kruskal':  # Categorical feature (Kruskal-Wallis)
            summary_data.append({
                'Column': var,
                'Test Type': 'Kruskal-Wallis',
                'H_statistic': results['h_stat'],
                'p_value': results['p_value'],
                'n_categories': results['n_categories'],
                'Significant': is_significant
            })
    # Handle old format (from statistical_tests_step1)
    elif 't_p' in results:  # Numerical test (binary target)
        summary_data.append({
            'Column': var,
            'Test Type': 'T-test & KS-test',
            't_p_value': results['t_p'],
            'ks_p_value': results['ks_p'],
            'Significant': is_significant
        })
    elif 'p_value' in results:  # ANOVA test (multi-class target)
        summary_data.append({
            'Column': var,
            'Test Type': 'ANOVA',
            'p_value': results['p_value'],
            'Significant': is_significant
        })
    elif 'p' in results:  # Chi-square test (categorical)
        summary_data.append({
            'Column': var,
            'Test Type': 'Chi-square',
            'p_value': results['p'],
            'Significant': is_significant
        })

summary_df = pd.DataFrame(summary_data)

# Display only significant columns
print("Summary of Significant Columns:")
print("=" * 80)
significant_df = summary_df[summary_df['Significant'] == True].copy()

if len(significant_df) > 0:
    # Sort by correlation strength for numerical features
    if 'Pearson_r' in significant_df.columns:
        significant_df['abs_correlation'] = significant_df['Pearson_r'].abs()
        significant_df = significant_df.sort_values('abs_correlation', ascending=False)
        significant_df = significant_df.drop('abs_correlation', axis=1)
    else:
        significant_df = significant_df.sort_values('Column')
    
    display(significant_df)
else:
    print("No significant columns found.")